# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All references to the data structure (record sets, fields, columns) are by their unique `@id`.

### Dataset Source
The dataset is described by a Croissant schema available at:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We load metadata and records from the FAIR^2 dataset using `mlcroissant`. The dataset describes 77 cancer survivors with second primary colorectal cancer.

**Key fields include clinical, pathological, and molecular variables.**

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata safely as an object
metadata = dataset.metadata
print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Number of authors: {len(metadata.author)}")
print(f"Version: {metadata.version}")

## 2. Data Overview

Let's review available **record sets** and their fields using `@id` as references.

> **Note**: For this dataset, record set and field metadata are embedded inside the Croissant schema. We'll use the `record_sets` method to enumerate them.

In [ ]:
# List all record sets available in the dataset (by @id and name):
record_sets = list(dataset.record_sets())
print(f"Number of record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name','<no name>')}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print(f"  Fields:")
        for field in fields:
            # Each field is a dict with '@id' and (maybe) name
            print(f"    - Field @id: {field['@id']}, name: {field.get('name','<no name>')}")
    print()

## 3. Data Extraction

Now we will load data from the **primary record set** into a DataFrame. All references use `@id` for reproducibility.

Below, we extract all record sets into Pandas DataFrames for further analysis.

In [ ]:
# Prepare to fetch all record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load {record_set_id}: {e}")

# For this dataset the main data is typically in a clinical table. Find the largest table (most rows), or select the main one by name:
if dataframes:
    # Use the record set with most rows as main table for illustration
    largest_rs = max(dataframes.items(), key=lambda x: x[1].shape[0])[0]
    print(f"\nMain analysis record set: {largest_rs}")
    print("Columns (Field @id):")
    print(list(dataframes[largest_rs].columns))
    display(dataframes[largest_rs].head())
else:
    print("No dataframes loaded!")

## 4. Exploratory Data Analysis (EDA)

Let's process a **numeric field**. For this demonstration, we'll look for a typical numeric field (e.g., age, duration, tumor size, etc.) among the columns. You'll need to update the `numeric_field_id` and `group_field_id` below according to your specific columns (by their `@id`).

We'll show filtering, normalization, and grouping using the field `@id`.

In [ ]:
# Identify a numeric field. We'll scan for a likely candidate, e.g. Age or a count/int/float field.
main_df = dataframes[largest_rs]

# Guess numeric columns if possible (float/int)
numeric_cols = main_df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Using numeric field (by @id): {numeric_field}")
else:
    print("No numeric field found.")
    # Use a known field id if documented
    numeric_field = None

# Example threshold
threshold = 60
if numeric_field:
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}: {filtered_df.shape[0]} records")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    filtered_df = main_df.copy()

# Find a likely categorical field for 'group by' (e.g. sex, group, anatomical site)
cat_cols = main_df.select_dtypes(include=['object', 'category']).columns.tolist()
possible_group_fields = [col for col in cat_cols if main_df[col].nunique() < 10 and main_df[col].nunique() > 1]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by field (by @id): {group_field}")
    if numeric_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        print(grouped_df)
else:
    group_field = None
    print("No suitable group field found.")

## 5. Visualization

Let's visualize the distribution of values for the selected numeric field and relationships with the group field. All axes/titles reflect the `@id` of the fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.show()
else:
    print('No numeric field selected for visualization.')

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR^2 clinical dataset using the `mlcroissant` library, referencing all data entities by their `@id`. We identified record sets, processed and visualized numeric and categorical data, and performed basic summary statistics. For further analysis, consider exploring specific clinical questions or hypothesis tests using the same `@id` field references.